### Simple JEV example

In [1]:
from typesafe_sdk import TypeSafeClient

# Initialize the client
client = TypeSafeClient(model="jev-latest")

# Shape the context as state for the System One model
state = {"Since this morning, none of our employees can access the CRM application. The system returns a 503 error."}

# Shape the question for the System One model
questions = {
    "issue_type": {
        "type": "choice",
        "instructions": "Classify the IT support ticket.",
        "criteria": {
            "incident": "Unplanned service disruption or failure.",
            "request": "Request for access or a new service.",
            "change": "Modification to an existing system."
        }
    },
    "requires_escalation": {
        "type": "noul",
        "instructions": (
            "Does this issue require immediate escalation?"
        ),
        "criteria": {
            "true": (
                "Critical disruption, widespread user impact, "
                "security incident, or business-critical "
                "system unavailable."
            ),
            "false": (
                "Minor or localized issue with limited "
                "business impact."
            )
        }
    },
    "urgency": {
        "type": "score",
        "instructions": "Assess the urgency of the IT support ticket.",
        "criteria": [
            "No immediate action required.",
            "Requires attention within the next few days.",
            "Requires attention today.",
            "Requires immediate action due to major disruption."
        ]
    }
}

# Invoking the system one model
response = client.system_one(state=state, questions=questions)

# Parsing the response per question
print(response.choices["issue_type"].choice, response.choices["issue_type"].confidence)
print(response.nouls["requires_escalation"].noul)
print(response.scores["urgency"].score, response.scores["urgency"].confidence)

incident 1.0
0.92
3.0 1.0


### Classification and Agent Routing

In [2]:
from typesafe_sdk import TypeSafeClient

# Initialize the client
client = TypeSafeClient(model="jev-latest")

# Shape the context as state for the System One model
state = "I want to check the status of my refund."

# Shape the question for the System One model
questions = {
    "route": {
        "type": "choice",
        "instructions": "Select the appropriate agent.",
        "criteria": {
            "billing": "Payments, refunds, and invoices.",
            "technical": "Technical issues and errors.",
            "general": "General inquiries."
        }
    }
}

# Invoking the system one model
response = client.system_one(state=state, questions=questions)

print(f"Choice of Agent: `{response.choices['route'].choice}` with confidence {response.choices['route'].confidence}")

Choice of Agent: `billing` with confidence 1.0


### Model Routing

In [3]:
from typesafe_sdk import TypeSafeClient

# Initialize the client
client = TypeSafeClient(model="jev-latest")

# Shape the context as state for the System One model
state = "Analyze the financial risks of acquiring a competitor."

# Shape the question for the System One model
questions = {
    "model_routing": {
        "type": "choice",
        "instructions": (
            "Select the appropriate LLM based on "
            "the request's reasoning complexity."
        ),
        "criteria": {
            "small_model": "Simple questions and basic text tasks.",
            "standard_model": "Moderate reasoning and analysis.",
            "reasoning_model": "Complex, multi-step reasoning tasks."
        }
    }
}

# Invoking the system one model
response = client.system_one(state=state, questions=questions)

print(f"Choice of Model: `{response.choices['model_routing'].choice}` with confidence {response.choices['model_routing'].confidence}")

Choice of Model: `reasoning_model` with confidence 0.52


### Detection, Guardrails, and Verification

In [4]:
from typesafe_sdk import TypeSafeClient

# Initialize the client
client = TypeSafeClient(model="jev-latest")

# Shape the context as state for the System One model
state = {
    "response": "The return policy is 60 days.",
    "context": "Customers can return products within 30 days."
}

# Shape the question for the System One model
questions = {
    "groundedness": {
        "type": "noul",
        "instructions": "Is the response supported by the context?",
        "criteria": {
            "true": "All factual claims are supported.",
            "false": "Contains unsupported or contradictory claims."
        }
    }
}

# Invoking the system one model
response = client.system_one(state=state, questions=questions)

print(f"The responsed is grounded by the contex: `{response.nouls['groundedness'].noul}` True")

The responsed is grounded by the contex: `0.02` True


### Scoring and Ranking

In [5]:
from typesafe_sdk import TypeSafeClient
from pprint import pprint


# Initialize the client
client = TypeSafeClient(model="jev-latest")

# Shape the context as state for the System One model
state = {
    "query": "How can I reset my password?",
    "documents": [
        "Steps to reset your account password.",
        "How to update your profile information.",
        "Troubleshooting password reset errors."
    ]
}

# Shape the question for the System One model
questions = {
    f"document_{i}_relevance": {
        "type": "score",
        "instructions": (
            f"Evaluate the relevance of document {i} "
            "to the user's query."
        ),
        "criteria": [
            "Not relevant.",
            "Partially relevant.",
            "Highly relevant."
        ]
    }
    for i in range(len(state["documents"]))
}

# Invoking the system one model
response = client.system_one(state=state, questions=questions)

pprint(response.answers, sort_dicts=False)

{'document_0_relevance': ScoreAnswer(type='score', score=2.0, confidence=1.0, legend={0: 'Not relevant.', 1: 'Partially relevant.', 2: 'Highly relevant.'}, probabilities={0: 0.0, 1: 0.0, 2: 1.0}),
 'document_1_relevance': ScoreAnswer(type='score', score=2.0, confidence=1.0, legend={0: 'Not relevant.', 1: 'Partially relevant.', 2: 'Highly relevant.'}, probabilities={0: 0.0, 1: 0.0, 2: 1.0}),
 'document_2_relevance': ScoreAnswer(type='score', score=0.17, confidence=0.76, legend={0: 'Not relevant.', 1: 'Partially relevant.', 2: 'Highly relevant.'}, probabilities={0: 0.84, 1: 0.16, 2: 0.0})}


## Appendix

In [6]:
# Token Usage & Cost
INPUT_COST_PER_1M = 0.042
OUTPUT_COST_PER_1M = 0.0

input_tokens = response.usage.input_tokens
output_tokens = response.usage.output_tokens
total_tokens = input_tokens + output_tokens

input_cost = (input_tokens / 1_000_000) * INPUT_COST_PER_1M
output_cost = (output_tokens / 1_000_000) * OUTPUT_COST_PER_1M
total_cost = input_cost + output_cost

print(f"Model:         {response.model}")
print(f"Input Tokens:  {input_tokens:,}")
print(f"Output Tokens: {output_tokens:,}")
print(f"Total Tokens:  {total_tokens:,}")
print(f"Total Cost:    ${total_cost:.8f}")

Model:         jev-1.14.0
Input Tokens:  471
Output Tokens: 58
Total Tokens:  529
Total Cost:    $0.00001978
